# Projeto (Seção 8) — enunciado

Foi aqui que paramos no projeto da seção anterior:

In [1]:
from datetime import date
from enum import Enum
from typing import Annotated, TypeVar
from uuid import UUID, uuid4
from pydantic import BaseModel, ConfigDict, Field, UUID4, field_serializer
from pydantic.alias_generators import to_camel


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


BoundedString = Annotated[str, Field(min_length=2, max_length=50)]

T = TypeVar('T')

BoundedList = Annotated[list[T], Field(min_length=1, max_length=5)]


class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=to_camel,
    )

    id_: UUID4 | None = Field(alias="id", default_factory=uuid4)
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType = Field(alias="type")
    is_electric: bool = False
    manufactured_date: date = Field(validation_alias="completionDate", ge=date(1980, 1, 1))
    base_msrp_usd: float = Field(
        validation_alias="msrpUSD",
        serialization_alias="baseMSRPUSD"
    )
    top_features: BoundedList[BoundedString] | None = None
    vin: BoundedString
    number_of_doors: int = Field(
        default=4,
        validation_alias="doors",
        ge=2,
        le=4,
        multiple_of=2,
    )
    registration_country: BoundedString | None = None
    license_plate: BoundedString | None = None

    @field_serializer("manufactured_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")

Faremos duas mudanças principais no modelo.

**Primeiro**, um campo novo para registrar quando o automóvel foi emplacado (registrado): `registration_date`, assim:
- logo depois de `registration_country` no modelo
- do tipo `date`
- opcional, com default `None`
- desserializado de e serializado para a versão camelCase do nome do campo
- não pode ser anterior a `manufactured_date`
- assim como `manufactured_date`, deve ser serializado em JSON no formato `YYYY/MM/DD` (dica: não é preciso um segundo serializador! A
  sintaxe é a mesma de aplicar um validador decorado a vários campos)

**Segundo**, queremos que `registration_country` só aceite valores de uma lista predeterminada de países.

Não vamos usar enum aqui, porque seriam valores demais. Vamos validar o nome do país contra um "banco de dados".

Neste exercício não usaremos banco de verdade: use o dicionário abaixo. As chaves são os valores de "entrada" aceitos para nomes de país,
e cada valor é uma tupla com o nome do país (formatado corretamente) e o código de 3 letras do país (o código não é usado agora, mas será
depois).

Crie um validador customizado para `registration_country` que verifique se o dado desserializado é uma das chaves do dicionário e
**substitua** o valor desserializado pelo nome do país (o primeiro elemento da tupla).

Por exemplo, se a entrada contiver:
```
{
    ...,
    "registrationCountry": "UK",
    ...
}
```
como o "banco" nos dá:
```
"uk": ("United Kingdom", "GBR")
```
o valor desserializado no modelo deve ser `United Kingdom`.

O validador deve comparar o nome do país pela versão em minúsculas e sem espaços da string — ou seja, entradas como `"UK"`, `"Uk"`,
`"uk "` devem todas casar com a chave `"uk"`.

Use um tipo anotado para isso — chame o novo tipo anotado de `Country`.

In [2]:
countries = {
    "australia": ("Australia", "AUS"),
    "canada": ("Canada", "CAN"),
    "china": ("China", "CHN"),
    "france": ("France", "FRA"),
    "germany": ("Germany", "DEU"),
    "india": ("India", "IND"),
    "mexico": ("Mexico", "MEX"),
    "norway": ("Norway", "NOR"),
    "pakistan": ("Pakistan", "PAK"),
    "san marino": ("San Marino", "SMR"),
    "sanmarino": ("San Marino", "SMR"),
    "spain": ("Spain", "ESP"),
    "sweden": ("Sweden", "SWE"),
    "united kingdom": ("United Kingdom", "GBR"),
    "uk": ("United Kingdom", "GBR"),
    "great britain": ("United Kingdom", "GBR"),
    "britain": ("United Kingdom", "GBR"),
    "us": ("United States of America", "USA"),
    "united states": ("United States of America", "USA"),
    "usa": ("United States of America", "USA"),
}

A lista é um pequeno recorte dos dados disponíveis em [https://www.iban.com/country-codes](https://www.iban.com/country-codes).

Fique à vontade para incluir o seu país na lista, se ele não estiver! (A solução comentada inclui o Brasil.)

Exemplo de entrada e saída serializada (dict e JSON):

In [3]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4 Competition xDrive",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "topFeatures": ["6 cylinders", "all-wheel drive", "convertible"],
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "us",
    "registrationDate": "2023-06-01",
    "licensePlate": "AAA-BBB"
}

expected_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4 Competition xDrive',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'topFeatures': ['6 cylinders', 'all-wheel drive', 'convertible'],
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'United States of America',
    'registrationDate': date(2023, 6, 1),
    'licensePlate': 'AAA-BBB'
}

expected_json_by_alias = '{"id":"c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7","manufacturer":"BMW","seriesName":"M4 Competition xDrive","type":"Convertible","isElectric":false,"manufacturedDate":"2023/01/01","baseMSRPUSD":93300.0,"topFeatures":["6 cylinders","all-wheel drive","convertible"],"vin":"1234567890","numberOfDoors":2,"registrationCountry":"United States of America","registrationDate":"2023/06/01","licensePlate":"AAA-BBB"}'

Não esqueça de testar também com dados inválidos para a data de registro e o nome do país!

➡️ Soluções: [`08_projeto_solucao_instrutor`](08_projeto_solucao_instrutor.ipynb) · [`09_projeto_solucao_comentada`](09_projeto_solucao_comentada.ipynb)